## 二手车价格预测


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from pandas.core.common import random_state
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.svm import SVR
import torch.nn as nn
import torch
import joblib

BASE_DIR = Path.cwd().resolve()
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'

### 数据处理


In [2]:
# 读数据
data = pd.read_csv(DATA_DIR / 'used_car_train.csv', low_memory=False, sep=r'\s+')
# 字符串处理
data = data.apply(lambda s: pd.Series([0 if x == '-' else x for x in s]))
data = data.apply(pd.to_numeric)
# 缺失值填充
data = data.fillna(data.mean())


### 特征工程

In [3]:
# 特征选择
X = data.drop(columns=['SaleID', 'name', 'price'])
y = data['price']
# 测训集分离
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 特征筛选

In [4]:
# 五折交叉Lasso
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lasso = LassoCV(cv=kf, max_iter=20000 , tol=1e-5, random_state=42)
lasso.fit(X_train_scaled, y_train)
# 特征筛选
selected_features = np.where(lasso.coef_ != 0)[0]
X_train_selected = X_train_scaled[:, selected_features]
X_test_selected = X_test_scaled[:, selected_features]

### 模型训练

In [7]:
# 线性回归
lr_model = LinearRegression()
lr_model.fit(X_train_selected, y_train)
joblib.dump(lr_model, MODEL_DIR/'lr_model.joblib', compress=9)

['/Users/alexdong/Programme/Daily_Learning/机器学习自学/实践项目/二手车价预测/models/lr_model.joblib']

In [8]:
# SVM
svm_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
svm_model.fit(X_train_selected, y_train)
joblib.dump(svm_model, MODEL_DIR/'svm_model.joblib', compress=9)

['/Users/alexdong/Programme/Daily_Learning/机器学习自学/实践项目/二手车价预测/models/svm_model.joblib']

In [9]:
# 随机森林
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_selected, y_train)
joblib.dump(rf_model, MODEL_DIR/'rf_model.joblib', compress=9)

['/Users/alexdong/Programme/Daily_Learning/机器学习自学/实践项目/二手车价预测/models/rf_model.joblib']

In [10]:
# 神经网络
class FNN(nn.Module):
    def __init__(self, input_dim):
        super(FNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 1)
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

    def fit(self, X:pd.DataFrame, y:pd.DataFrame, epochs=2001, verbose=True):
        self.train()
        y = np.array(y)
        inputs = torch.from_numpy(X).float()
        targets = torch.from_numpy(y).float().view(-1, 1)
        for epoch in range(epochs):
            outputs = self.forward(inputs)
            loss = self.criterion(outputs, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            if verbose and epoch % 200 == 0:
                print(f'Epoch {epoch}: Loss: {loss.item():.4f}')
        self.eval()

mlp_model = FNN(X_train_selected.shape[1])
mlp_model.fit(X_train_selected, y_train)
joblib.dump(mlp_model, MODEL_DIR/'mlp_model.joblib', compress=9)

Epoch 0: Loss: 88143648.0000
Epoch 200: Loss: 10527419.0000
Epoch 400: Loss: 7862426.0000
Epoch 600: Loss: 5734663.0000
Epoch 800: Loss: 4659477.5000
Epoch 1000: Loss: 4211488.5000
Epoch 1200: Loss: 3839967.2500
Epoch 1400: Loss: 3697049.0000
Epoch 1600: Loss: 3499652.2500
Epoch 1800: Loss: 3432497.5000
Epoch 2000: Loss: 3323674.0000


['/Users/alexdong/Programme/Daily_Learning/机器学习自学/实践项目/二手车价预测/models/mlp_model.joblib']

### 模型评估

In [5]:
# lr
lr_model = joblib.load(MODEL_DIR/'lr_model.joblib')
pred = lr_model.predict(X_test_selected)
mae = mean_absolute_error(y_test, pred)
print(f'线性回归 MAE: {mae}')

线性回归 MAE: 2714.425275808949


In [6]:
# svm
svm_model = joblib.load(MODEL_DIR/'svm_model.joblib')
pred = svm_model.predict(X_test_selected)
mae = mean_absolute_error(y_test, pred)
print(f'支持向量机 MAE: {mae}')

支持向量机 MAE: 903.6236088641609


In [7]:
# rf
rf_model = joblib.load(MODEL_DIR/'rf_model.joblib')
pred = rf_model.predict(X_test_selected)
mae = mean_absolute_error(y_test, pred)
print(f'随机森林 MAE: {mae}')

随机森林 MAE: 598.4457664633329


In [8]:
# mlp
mlp_model = joblib.load(MODEL_DIR/'mlp_model.joblib')
with torch.no_grad():
    inputs = torch.from_numpy(X_test_selected).float()
    pred = mlp_model(inputs).numpy().flatten()
mae = mean_absolute_error(y_test, pred)
print(f'神经网络 MAE: {mae}')

AttributeError: module '__main__' has no attribute 'FNN'

### stacking

In [17]:
# 准备
class StackingFNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.01)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

    def fit(self, X, y, epochs=2001, verbose=True):
        self.train()
        y = np.array(y)
        inputs = torch.from_numpy(X).float()
        targets = torch.tensor(y).float()
        for epoch in range(epochs):
            outputs = self.forward(inputs)
            loss = self.criterion(outputs, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            if verbose and epoch % 200 == 0:
                print(f'Epoch {epoch}: Loss: {loss.item():.4f}')
        self.eval()

stacking_X_train = np.column_stack((lr_model.predict(X_train_selected),
                                    svm_model.predict(X_train_selected),
                                    rf_model.predict(X_train_selected)))
stacking_X_test = np.column_stack((lr_model.predict(X_test_selected),
                                   svm_model.predict(X_test_selected),
                                   rf_model.predict(X_test_selected)))

In [ ]:
# 训练
stacking_model = StackingFNN(3)
stacking_model.fit(stacking_X_train, y_train)
joblib.dump(stacking_model, MODEL_DIR/'stacking_model.joblib', compress=9)

/Users/alexdong/Programme/Daily_Learning/.venv/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([120000])) that is different to the input size (torch.Size([120000, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
# 评估
with torch.no_grad():
    inputs = torch.from_numpy(X_test_selected).float()
    pred = stacking_model.predict(inputs)
    mae = mean_absolute_error(y_test, pred)
    print(f'Stacking MAE: {mae}')